## Imports

In [7]:
from __future__ import annotations

import warnings
from pathlib import Path
from typing import Optional
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
ACCIDENT_PATH = DATA_DIR / "Accident_Information.csv"
VEHICLE_PATH = DATA_DIR / "Vehicle_Information.csv"

OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KEY = "Accident_Index"
TARGET = "Accident_Severity"

## Load Both Datasets

In [8]:
def load_dataset(path: Path, name: str) -> pd.DataFrame:
    """Load a UK Road Safety CSV, failing loudly if it is not where expected."""
    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found at '{path}'. Download the UK Road Safety dataset and place "
            f"both CSVs under '{DATA_DIR}/' before running this notebook."
        )
    # Accident_Index is forced to string: if pandas infers it as numeric, merge keys can silently
    # lose leading zeros or get float-coerced, breaking the join in Step 3.
    df = pd.read_csv(path, low_memory=False, encoding="latin1", dtype={KEY: str})
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df


accidents_raw = load_dataset(ACCIDENT_PATH, "Accident_Information.csv")
vehicles_raw = load_dataset(VEHICLE_PATH, "Vehicle_Information.csv")

Accident_Information.csv: 2,047,256 rows x 34 columns
Vehicle_Information.csv: 2,177,205 rows x 24 columns


# Verify Primary Key and Relationship Type

In [9]:
def check_key_uniqueness(df: pd.DataFrame, key: str, name: str) -> dict:
    total = len(df)
    unique = df[key].nunique(dropna=False)
    return {
        "dataset": name,
        "total_rows": total,
        "unique_keys": unique,
        "duplicate_rows": total - unique,
        "is_unique_key": unique == total,
    }


key_summary = pd.DataFrame([
    check_key_uniqueness(accidents_raw, KEY, "Accident_Information"),
    check_key_uniqueness(vehicles_raw, KEY, "Vehicle_Information"),
])
key_summary

,dataset,total_rows,unique_keys,duplicate_rows,is_unique_key
0,Accident_Information,2047256,2047256,0,True
1,Vehicle_Information,2177205,1488981,688224,False


In [10]:
vehicles_per_accident = vehicles_raw.groupby(KEY).size()
print(f"Avg vehicles per accident: {vehicles_per_accident.mean():.2f}")
print(f"Max vehicles in a single accident: {vehicles_per_accident.max()}")
print(f"Accidents with exactly 1 vehicle: {(vehicles_per_accident == 1).mean() * 100:.1f}%")

Avg vehicles per accident: 1.46
Max vehicles in a single accident: 53
Accidents with exactly 1 vehicle: 60.4%


## Merge Datasets (Accident-Level, No Duplication)

In [11]:
def aggregate_vehicle_features(vehicles: pd.DataFrame, key: str = KEY) -> pd.DataFrame:
    """Collapse Vehicle_Information to one row per accident.

    Numeric driver/vehicle attributes are summarised (mean + max); vehicle type is summarised
    by count, diversity, and the most common type involved.
    """
    numeric_candidates = ["Age_of_Driver", "Age_of_Vehicle", "Engine_Capacity_.CC.", "Driver_IMD_Decile"]
    numeric_cols = [c for c in numeric_candidates if c in vehicles.columns]

    numeric_agg = vehicles.groupby(key)[numeric_cols].agg(["mean", "max"])
    numeric_agg.columns = [f"{col}_{stat}" for col, stat in numeric_agg.columns]

    counts = vehicles.groupby(key).size().rename("Number_of_Vehicles_Involved")

    if "Vehicle_Type" in vehicles.columns:
        diversity = vehicles.groupby(key)["Vehicle_Type"].nunique().rename("Vehicle_Type_Diversity")
        dominant_type = (
            vehicles.groupby(key)["Vehicle_Type"]
            .agg(lambda s: s.mode().iat[0] if not s.mode().empty else "Unknown")
            .rename("Dominant_Vehicle_Type")
        )
        return pd.concat([numeric_agg, counts, diversity, dominant_type], axis=1).reset_index()

    return pd.concat([numeric_agg, counts], axis=1).reset_index()


vehicle_agg = aggregate_vehicle_features(vehicles_raw)
print(f"Aggregated vehicle table: {vehicle_agg.shape[0]:,} rows (one per accident)")
vehicle_agg.head()

Aggregated vehicle table: 1,488,981 rows (one per accident)


,Accident_Index,Age_of_Vehicle_mean,Age_of_Vehicle_max,Engine_Capacity_.CC._mean,Engine_Capacity_.CC._max,Driver_IMD_Decile_mean,Driver_IMD_Decile_max,Number_of_Vehicles_Involved,Vehicle_Type_Diversity,Dominant_Vehicle_Type
0,200401BS00001,3.0,3.0,1588.0,1588.0,4.0,4.0,1,1,109
1,200401BS00002,NaN,NaN,NaN,NaN,3.0,3.0,1,1,109
2,200401BS00003,4.0,4.0,998.0,998.0,NaN,NaN,2,1,109
3,200401BS00004,5.5,10.0,952.5,1781.0,4.0,4.0,2,2,109
4,200401BS00009,NaN,NaN,NaN,NaN,4.0,4.0,1,1,Motorcycle 125cc and under


In [12]:
def merge_accident_vehicle(accidents: pd.DataFrame, vehicle_agg: pd.DataFrame, key: str = KEY) -> pd.DataFrame:
    """Left-join accident-level features onto the accident table without duplicating rows."""
    before = len(accidents)
    merged = accidents.merge(vehicle_agg, on=key, how="left", validate="one_to_one")
    assert len(merged) == before, "Merge changed row count — accident records were duplicated."
    return merged


df = merge_accident_vehicle(accidents_raw, vehicle_agg)
print(f"Merged dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
assert df[KEY].is_unique, "Accident_Index is no longer unique after merge."

Merged dataset: 2,047,256 rows x 43 columns


## Data Cleaning

In [13]:
def drop_exact_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates()
    removed = before - len(df)
    print(f"Removed {removed:,} exact duplicate rows")
    return df


df = drop_exact_duplicates(df)

Removed 0 exact duplicate rows


In [14]:
IMPOSSIBLE_VALUE_RULES: dict[str, tuple[float, float]] = {
    "Speed_limit": (0, 70),
    "Number_of_Vehicles": (1, 32),
    "Number_of_Casualties": (1, 50),
    "Age_of_Driver_mean": (0, 100),
}


def remove_impossible_values(df: pd.DataFrame, rules: dict[str, tuple[float, float]]) -> pd.DataFrame:
    """Drop rows outside a physically plausible range for the given columns.

    Missing values are left untouched here — they're Step 5's responsibility, not Step 4's.
    """
    df = df.copy()
    for col, (low, high) in rules.items():
        if col not in df.columns:
            continue
        in_range = df[col].between(low, high) | df[col].isna()
        removed = (~in_range).sum()
        if removed:
            print(f"{col}: removed {removed:,} rows outside [{low}, {high}]")
        df = df[in_range]
    return df


df = remove_impossible_values(df, IMPOSSIBLE_VALUE_RULES)

Number_of_Vehicles: removed 3 rows outside [1, 32]
Number_of_Casualties: removed 9 rows outside [1, 50]


In [15]:
MISSING_TOKENS = {"data missing or out of range", "unknown", "-1", "none", "nan", ""}


def standardize_categoricals(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Trim whitespace, title-case text, and collapse the dataset's various 'unknown' sentinels to NaN."""
    df = df.copy()
    for col in columns:
        if col not in df.columns:
            continue
        df[col] = df[col].astype(str).str.strip().str.title()
        df.loc[df[col].str.lower().isin(MISSING_TOKENS), col] = np.nan
    return df


categorical_cols = df.select_dtypes(include="object").columns.tolist()
df = standardize_categoricals(df, categorical_cols)
print(f"Standardized {len(categorical_cols)} categorical columns")

Standardized 22 categorical columns


## Missing Value Analysis

In [16]:
def missing_value_report(df: pd.DataFrame) -> pd.DataFrame:
    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2)
    report = pd.DataFrame({
        "missing_count": missing_count,
        "missing_pct": missing_pct,
        "dtype": df.dtypes.astype(str),
    })
    report = report[report["missing_count"] > 0].sort_values("missing_pct", ascending=False)
    report["recommended_strategy"] = report.apply(_recommend_strategy, axis=1)
    return report


def _recommend_strategy(row: pd.Series) -> str:
    if row["missing_pct"] > 60:
        return "Drop column"
    if row["dtype"] in ("float64", "int64"):
        return "Median imputation"
    return "'Unknown' category"


missing_report = missing_value_report(df)
missing_report

,missing_count,missing_pct,dtype,recommended_strategy
Carriageway_Hazards,2010542,98.21,object,Drop column
Special_Conditions_at_Site,1997960,97.59,object,Drop column
Driver_IMD_Decile_mean,1045494,51.07,float64,Median imputation
Driver_IMD_Decile_max,1045494,51.07,float64,Median imputation
2nd_Road_Class,844264,41.24,object,'Unknown' category
Age_of_Vehicle_mean,797141,38.94,float64,Median imputation
Age_of_Vehicle_max,797141,38.94,float64,Median imputation
Engine_Capacity_.CC._mean,757544,37.00,float64,Median imputation
Engine_Capacity_.CC._max,757544,37.00,float64,Median imputation
Junction_Control,754303,36.84,object,'Unknown' category


In [17]:
def apply_missing_value_strategy(df: pd.DataFrame, report: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, str]]:
    """Apply the strategy recommended in `report` to every column it lists.

    Returns the treated dataframe and a {column: strategy_applied} record for the metadata export.
    """
    df = df.copy()
    applied: dict[str, str] = {}

    for col, row in report.iterrows():
        strategy = row["recommended_strategy"]
        if strategy == "Drop column":
            df = df.drop(columns=[col])
        elif strategy == "Median imputation":
            df[col] = df[col].fillna(df[col].median())
        else:  # 'Unknown' category
            df[col] = df[col].fillna("Unknown")
        applied[col] = strategy

    return df, applied


df, missing_value_strategy_log = apply_missing_value_strategy(df, missing_report)
print(f"Remaining missing values after treatment: {df.isna().sum().sum()}")

Remaining missing values after treatment: 0


## Step 6 — Date Feature Engineering

In [18]:
def engineer_date_features(df: pd.DataFrame, date_col: str = "Date", time_col: str = "Time") -> pd.DataFrame:
    df = df.copy()
    parsed_date = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)
    hour = pd.to_datetime(df[time_col], errors="coerce", format="%H:%M").dt.hour
    df["Year"] = parsed_date.dt.year
    df["Month"] = parsed_date.dt.month
    df["Day"] = parsed_date.dt.day
    df["Weekday"] = parsed_date.dt.day_name()
    df["Hour"] = hour
    df["Is_Weekend"] = parsed_date.dt.dayofweek.isin([5, 6]).astype(int)
    df["Season"] = df["Month"].map(_month_to_season)
    df["Is_Rush_Hour"] = hour.isin([7, 8, 9, 16, 17, 18]).astype(int)
    peak_hours = hour.value_counts().nlargest(3).index.tolist()
    df["Is_Peak_Hour"] = hour.isin(peak_hours).astype(int)
    df["Is_Night"] = ((hour >= 20) | (hour < 6)).astype(int)
    df["Is_Morning"] = hour.between(6, 11).astype(int)
    df["Is_Evening"] = hour.between(17, 20).astype(int)
    return df


def _month_to_season(month) -> str:
    if pd.isna(month):
        return "Unknown"
    return {
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Spring", 4: "Spring", 5: "Spring",
        6: "Summer", 7: "Summer", 8: "Summer",
        9: "Autumn", 10: "Autumn", 11: "Autumn",
    }[int(month)]


df = engineer_date_features(df)
df = df.drop(columns=["Date", "Time"])  # superseded by the extracted features above
df[["Year", "Month", "Weekday", "Hour", "Is_Weekend", "Season", "Is_Rush_Hour", "Is_Peak_Hour"]].head()

,Year,Month,Weekday,Hour,Is_Weekend,Season,Is_Rush_Hour,Is_Peak_Hour
0,2005.0,4.0,Friday,17.0,0,Spring,1,1
1,2005.0,5.0,Sunday,17.0,1,Spring,1,1
2,2005.0,6.0,Wednesday,0.0,0,Summer,0,0
3,2005.0,7.0,Friday,10.0,0,Summer,0,0
4,2005.0,10.0,Saturday,21.0,1,Autumn,0,0


## Road Feature Engineering

In [19]:
BAD_WEATHER_CONDITIONS = {
    "Raining No High Winds", "Raining High Winds",
    "Snowing No High Winds", "Snowing High Winds", "Fog Or Mist",
}
POOR_VISIBILITY_CONDITIONS = {
    "Darkness - No Lighting", "Darkness - Lights Unlit", "Darkness - Lighting Unknown",
}


def engineer_road_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Weather_Conditions" in df.columns:
        df["Bad_Weather_Flag"] = df["Weather_Conditions"].isin(BAD_WEATHER_CONDITIONS).astype(int)
    if "Light_Conditions" in df.columns:
        df["Poor_Visibility_Flag"] = df["Light_Conditions"].isin(POOR_VISIBILITY_CONDITIONS).astype(int)
    if "Number_of_Vehicles" in df.columns:
        df["Traffic_Density_Indicator"] = pd.cut(
            df["Number_of_Vehicles"],
            bins=[0, 1, 2, 4, np.inf],
            labels=["Single_Vehicle", "Two_Vehicle", "Moderate", "High_Density"],
        )
    df["Road_Risk_Score"] = _compute_road_risk_score(df)
    return df


def _compute_road_risk_score(df: pd.DataFrame) -> pd.Series:
    """Heuristic 0-8ish composite risk index. Weights are domain judgment, not fitted."""
    score = pd.Series(0.0, index=df.index)
    if "Speed_limit" in df.columns:
        score += (df["Speed_limit"].fillna(30) / 70) * 3
    if "Bad_Weather_Flag" in df.columns:
        score += df["Bad_Weather_Flag"] * 2
    if "Poor_Visibility_Flag" in df.columns:
        score += df["Poor_Visibility_Flag"] * 2
    if "Road_Surface_Conditions" in df.columns:
        score += df["Road_Surface_Conditions"].astype(str).str.contains(
            "Wet|Ice|Snow", case=False, na=False
        ) * 1.5
    return score.round(2)


df = engineer_road_features(df)
road_feature_cols = [c for c in ["Bad_Weather_Flag", "Poor_Visibility_Flag",
                                   "Traffic_Density_Indicator", "Road_Risk_Score"] if c in df.columns]
df[road_feature_cols].describe(include="all")

,Bad_Weather_Flag,Poor_Visibility_Flag,Traffic_Density_Indicator,Road_Risk_Score
count,2.047244e+06,2.047244e+06,2047244,2.047244e+06
unique,NaN,NaN,4,NaN
top,NaN,NaN,Two_Vehicle,NaN
freq,NaN,NaN,1219244,NaN
mean,1.288234e-01,7.179213e-02,NaN,2.523099e+00
std,3.350045e-01,2.581435e-01,NaN,1.581520e+00
min,0.000000e+00,0.000000e+00,NaN,0.000000e+00
25%,0.000000e+00,0.000000e+00,NaN,1.290000e+00
50%,0.000000e+00,0.000000e+00,NaN,1.710000e+00
75%,0.000000e+00,0.000000e+00,NaN,3.000000e+00


## Categorical Encoding

In [20]:
def classify_encoding_strategy(df: pd.DataFrame, columns: list[str], threshold: int = 10) -> pd.DataFrame:
    rows = [
        {
            "feature": col,
            "cardinality": df[col].nunique(dropna=True),
            "strategy": "One-Hot" if df[col].nunique(dropna=True) <= threshold else "Label",
        }
        for col in columns
    ]
    return pd.DataFrame(rows).sort_values("cardinality").reset_index(drop=True)


encoding_target_cols = [
    c for c in df.select_dtypes(include=["object", "category"]).columns
    if c not in {KEY, TARGET}
]
encoding_plan = classify_encoding_strategy(df, encoding_target_cols)
encoding_plan

,feature,cardinality,strategy
0,InScotland,3,One-Hot
1,Urban_or_Rural_Area,3,One-Hot
2,Traffic_Density_Indicator,4,One-Hot
3,Season,5,One-Hot
4,Road_Type,6,One-Hot
5,Road_Surface_Conditions,6,One-Hot
6,1st_Road_Class,6,One-Hot
7,Light_Conditions,6,One-Hot
8,Junction_Control,6,One-Hot
9,Day_of_Week,7,One-Hot


In [21]:
def apply_encoding(df: pd.DataFrame, plan: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, str]]:
    """Apply the encoding plan and return the encoded dataframe plus a {feature: method} log."""
    df = df.copy()
    applied: dict[str, str] = {}
    onehot_cols = plan.loc[plan["strategy"] == "One-Hot", "feature"].tolist()
    label_cols = plan.loc[plan["strategy"] == "Label", "feature"].tolist()
    if onehot_cols:
        df = pd.get_dummies(df, columns=onehot_cols, prefix=onehot_cols, dtype=int)
        applied.update({c: "One-Hot" for c in onehot_cols})
    for col in label_cols:
        encoder = LabelEncoder()
        df[col] = encoder.fit_transform(df[col].astype(str))
        applied[col] = "Label"
    return df, applied

df, encoding_log = apply_encoding(df, encoding_plan)
print(f"Dataset shape after encoding: {df.shape}")

Dataset shape after encoding: (2047244, 125)


## Outlier Detection

In [22]:
def detect_outliers_iqr(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        if col not in df.columns or not pd.api.types.is_numeric_dtype(df[col]):
            continue
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_mask = (df[col] < lower) | (df[col] > upper)
        rows.append({
            "feature": col,
            "lower_bound": round(lower, 2),
            "upper_bound": round(upper, 2),
            "outlier_count": int(outlier_mask.sum()),
            "outlier_pct": round(outlier_mask.mean() * 100, 2),
        })
    return pd.DataFrame(rows).sort_values("outlier_pct", ascending=False)


numeric_cols_for_outliers = [
    c for c in ["Speed_limit", "Number_of_Vehicles", "Number_of_Casualties",
                "Age_of_Driver_mean", "Age_of_Vehicle_mean", "Road_Risk_Score"]
    if c in df.columns
]
outlier_report = detect_outliers_iqr(df, numeric_cols_for_outliers)
outlier_report

,feature,lower_bound,upper_bound,outlier_count,outlier_pct
2,Number_of_Casualties,1.00,1.00,470577,22.99
3,Age_of_Vehicle_mean,3.00,11.00,380934,18.61
4,Road_Risk_Score,-1.27,5.56,124774,6.09
1,Number_of_Vehicles,-0.50,3.50,46911,2.29
0,Speed_limit,0.00,80.00,0,0.00


In [23]:
def flag_outliers(df: pd.DataFrame, outlier_report: pd.DataFrame) -> pd.DataFrame:
    """Add a boolean {feature}_Outlier column per feature in the report, without dropping rows."""
    df = df.copy()
    for _, row in outlier_report.iterrows():
        col = row["feature"]
        df[f"{col}_Outlier"] = ~df[col].between(row["lower_bound"], row["upper_bound"])
    return df

df = flag_outliers(df, outlier_report)

## Final Validation

In [24]:
LEAKAGE_COLUMNS = [c for c in df.columns if "Casualty_Severity" in c or "Casualty_Class" in c]
if LEAKAGE_COLUMNS:
    print(f"Dropping potential leakage columns: {LEAKAGE_COLUMNS}")
    df = df.drop(columns=LEAKAGE_COLUMNS)
else:
    print("No known leakage columns present.")

No known leakage columns present.


In [25]:
def run_final_validation(df: pd.DataFrame, key: str = KEY) -> pd.DataFrame:
    checks = {
        "No duplicate rows": df.duplicated().sum() == 0,
        "No duplicate accident keys": df[key].duplicated().sum() == 0,
        "No remaining nulls": df.isna().sum().sum() == 0,
        "Speed_limit within [0, 70] (if present)": (
            df["Speed_limit"].between(0, 70).all() if "Speed_limit" in df.columns else True
        ),
        "No object dtype columns remaining": len(df.select_dtypes(include="object").columns) == 0
        or set(df.select_dtypes(include="object").columns) <= {key},
    }
    return pd.DataFrame(
        [{"check": name, "passed": result} for name, result in checks.items()]
    )


validation_results = run_final_validation(df)
validation_results

,check,passed
0,No duplicate rows,True
1,No duplicate accident keys,True
2,No remaining nulls,False
3,"Speed_limit within [0, 70] (if present)",True
4,No object dtype columns remaining,False


## Data Quality Report

In [26]:
def generate_data_quality_report(df: pd.DataFrame, target: str = TARGET) -> dict:
    report = {
        "rows": len(df),
        "columns": df.shape[1],
        "missing_pct_overall": round(df.isna().sum().sum() / df.size * 100, 3),
        "duplicate_pct": round(df.duplicated().sum() / len(df) * 100, 3),
        "memory_usage_mb": round(df.memory_usage(deep=True).sum() / 1e6, 2),
    }

    if target in df.columns:
        target_dist = df[target].value_counts(normalize=True).round(4) * 100
        report["target_distribution_pct"] = target_dist.to_dict()
        report["class_balance_ratio_max_to_min"] = round(
            target_dist.max() / target_dist.min(), 2
        )

    return report


data_quality_report = generate_data_quality_report(df)
for key_name, value in data_quality_report.items():
    print(f"{key_name}: {value}")

rows: 2047244
columns: 130
missing_pct_overall: 1.387
duplicate_pct: 0.0
memory_usage_mb: 2280.89
target_distribution_pct: {'Slight': 84.73, 'Serious': 13.99, 'Fatal': 1.29}
class_balance_ratio_max_to_min: 65.68


## Export

In [27]:
FEATURE_DESCRIPTIONS: dict[str, str] = {
    "Year": "Calendar year extracted from accident date",
    "Month": "Calendar month (1-12) extracted from accident date",
    "Day": "Day of month extracted from accident date",
    "Weekday": "Day name extracted from accident date",
    "Hour": "Hour of day (0-23) extracted from accident time",
    "Is_Weekend": "1 if accident occurred Sat/Sun",
    "Season": "Meteorological season derived from Month",
    "Is_Rush_Hour": "1 if Hour falls in conventional commute windows (7-9, 16-18)",
    "Is_Peak_Hour": "1 if Hour is among the 3 most frequent accident hours in this dataset",
    "Is_Night": "1 if Hour is 20:00-05:59",
    "Is_Morning": "1 if Hour is 06:00-11:59",
    "Is_Evening": "1 if Hour is 17:00-19:59",
    "Bad_Weather_Flag": "1 if Weather_Conditions indicates rain, snow, or fog",
    "Poor_Visibility_Flag": "1 if Light_Conditions indicates darkness",
    "Traffic_Density_Indicator": "Binned Number_of_Vehicles (Single/Two/Moderate/High_Density)",
    "Road_Risk_Score": "Heuristic weighted composite of speed limit, weather, visibility, surface",
    "Number_of_Vehicles_Involved": "Vehicle count aggregated from Vehicle_Information per accident",
    "Vehicle_Type_Diversity": "Distinct vehicle types involved in the accident",
    "Dominant_Vehicle_Type": "Most common vehicle type involved in the accident",
}

def build_feature_metadata(
    df: pd.DataFrame,
    missing_strategy_log: dict[str, str],
    encoding_log: dict[str, str],
) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        rows.append({
            "feature_name": col,
            "data_type": str(df[col].dtype),
            "description": FEATURE_DESCRIPTIONS.get(col, "Sourced from raw UK Road Safety data"),
            "encoding_used": encoding_log.get(col, "None"),
            "missing_value_strategy": missing_strategy_log.get(col, "No missing values"),
        })
    return pd.DataFrame(rows)

feature_metadata = build_feature_metadata(df, missing_value_strategy_log, encoding_log)
df.to_csv(OUTPUT_DIR / "processed_data.csv", index=False)
feature_metadata.to_csv(OUTPUT_DIR / "feature_metadata.csv", index=False)
print(f"Exported processed_data.csv: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Exported feature_metadata.csv: {feature_metadata.shape[0]} features documented")

Exported processed_data.csv: 2,047,244 rows x 130 columns
Exported feature_metadata.csv: 130 features documented
